In [11]:
%run -n "/Users/alejandrogomez-paz/Desktop/RAG/rag_code/old_RAG.ipynb"

Total chunks: 41
Example meta: {'source': 'OSHA — Radiation Emergency Preparedness and Response', 'page': 'Health and Safety Hazards during Radiation Emergencies', 'chunk_id': 'osha_rep_hazards'}


/opt/anaconda3/envs/py_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 2/2 [00:01<00:00,  1.14it/s]


FAISS index size: 41
The threshold for ARS depends on the dose received and the rate of exposure.

- At approximately 200 rem, transient skin erythema may appear within 2-24 hours.
- At 9000 rem or greater (whole body acute), death can occur from cerebral/vascular breakdown.
- The LD50/30 — the whole body dose that results in lethality to 50% of an exposed population within 30 days — is estimated at 9000 rem.

Doses of 100 rad (1 Gy) or greater raise concern for acute death risk.


In [16]:
import json
import sys
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/local_LLM")
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1")

from rad_ai import query
from benchmark_golden_pairs_1 import golden_pairs, should_refuse_map, answer_keys
import time

'''
Quality:
  CoP - Context Precision: fraction of retrieved chunks that are golden
  CiP - Citation Precision: fraction of cited sources that are correct
  RR  - Refusal Rate: fraction of correct refusal decisions (maximize)
  HR  - Hallucination Rate: fraction of answers with unsupported claims (minimize)
Latency:
  RT  - mean retrieval time (s)
'''

CORPUS_PATH = r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1/corpus_1.jsonl"
metas = [
    {"source": r["source"], "page": r["section"], "chunk_id": r["chunk_id"]}
    for r in (json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8"))
    if r.get("record_type") == "chunk"
]

cid2sp   = {m["chunk_id"]: f'{m["source"]} | page {m["page"]}' for m in metas}
golden_sp = {q: {cid2sp[c] for c in ids} for q, ids in golden_pairs.items()}


def get_citation(hit) -> str:
    return f'{hit["source"]} | page {hit["page"]}'

def context_precision(query, retrieved_hits):
    if not retrieved_hits: return 0.0
    g = golden_sp.get(query, set())
    return sum(1 for h in retrieved_hits if get_citation(h) in g) / len(retrieved_hits)

def context_recall(query, retrieved_hits):        # Hit@k — the honest single-golden signal
    g = golden_sp.get(query, set())
    if not g: return None
    got = {get_citation(h) for h in retrieved_hits}
    return len(got & g) / len(g)

def citation_precision(cited_ids, golden_ids):    # cited_ids already (source,page) strings
    if not cited_ids: return 0.0
    return sum(1 for c in cited_ids if c in golden_ids) / len(cited_ids)

def refusal_score(did_refuse, should_refuse):
    correct = sum(1 for d, s in zip(did_refuse, should_refuse) if d == s)
    return correct / len(should_refuse)

def hallucination_rate(judgments):
    """judgments: list of bools, True = answer contained hallucination.
    v0: with an LLM judge."""
    if not judgments:
        return 0.0
    return sum(judgments) / len(judgments)

def quality_score(s):
    return (0.25 * s['COP']
          + 0.25 * s['CIP']
          + 0.25 * s['RR']          # already "higher = better"
          + 0.25 * (1 - s['HR']))

def context_tokens(hits) -> int:
    """Estimate tokens in the context sent to the LLM.
    Rebuilds the same [SOURCE: ...] blocks rag_answer sends, ~4 chars/token."""
    blocks = [
        f"[SOURCE: {h['source']} | page {h['page']} | score 0.000]\n{h['text']}"
        for h in hits
    ]
    context = "\n\n---\n\n".join(blocks)
    return len(context) // 4


def judge_hallucination(q, answer, chunks):
    """LLM-as-judge. Returns True if the answer makes claims not supported by the retrieved chunks."""
    import json, re
    from rad_ai import query

    # a correct refusal cannot hallucinate
    if isinstance(answer, dict) and answer.get('refused'):
        return False

    # pull answer text
    if isinstance(answer, dict):
        ans = answer.get('answer') or answer.get('text') or answer.get('response') or json.dumps(answer)
    else:
        ans = str(answer)
    if not str(ans).strip():
        return False

    # build context from retrieved hits
    def chunk_text(h):
        for k in ('text', 'content', 'chunk', 'body', 'passage'):
            if isinstance(h, dict) and h.get(k):
                return str(h[k])
        return str(h)
    context = "\n\n".join(f"[{i+1}] {chunk_text(h)}" for i, h in enumerate(chunks)) or "(no context)"

    prompt = (
        "You are a strict factuality judge for a retrieval-augmented QA system. "
        "Given a QUESTION, the retrieved CONTEXT, and an ANSWER, decide whether the ANSWER "
        "contains a hallucination: any factual claim NOT supported by the CONTEXT. "
        "Judge support only against the CONTEXT, not outside knowledge. A refusal is NOT a hallucination.\n"
        'Respond with ONLY JSON: {"hallucinated": true|false}\n\n'
        f"QUESTION:\n{q}\n\nCONTEXT:\n{context}\n\nANSWER:\n{ans}\n"
    )

    try:
        raw = query(prompt)
        raw = raw if isinstance(raw, str) else str(raw)
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            return bool(json.loads(m.group(0)).get('hallucinated', False))
        return bool(re.search(r'\btrue\b|\byes\b', raw, re.I))
    except Exception:
        return False  # judge failed -> don't penalize
    
REFUSAL = "don't have enough information"

def benchmark_v0(k=5):
    latencies, cop_list, crec_list, cip_list, ctx_toks = [], [], [], [], []
    did_refuse, should_refuse, halluc = [], [], []

    for q, sr in should_refuse_map.items():
        t0 = time.time()
        hits = [{"source": m["source"], "page": m["page"],
                 "chunk_id": m["chunk_id"], "text": txt}
                for _, txt, m in retrieve(q, k=k)]
        latencies.append(time.time() - t0)

        ctx_toks.append(context_tokens(hits))

        answer = rag_answer(q, k=k)
        refused = REFUSAL in answer.lower()

        cop_list.append(context_precision(q, hits))
        r = context_recall(q, hits)
        if r is not None: crec_list.append(r)

        if not sr:  # CiP + hallucination judged on answerable queries only
            cited = [get_citation(h) for h in hits
                     if h["source"].lower() in answer.lower()]
            cip_list.append(citation_precision(cited, golden_sp.get(q, set())))
            halluc.append(False if refused else judge_hallucination(q, answer, hits))

        did_refuse.append(refused)
        should_refuse.append(sr)

    scores = {
        'COP': sum(cop_list)/len(cop_list),
        'REC': sum(crec_list)/len(crec_list),
        'CIP': sum(cip_list)/len(cip_list),
        'RR':  refusal_score(did_refuse, should_refuse),
        'HR':  hallucination_rate(halluc),
        'RT':  sum(latencies)/len(latencies),
        'CTX_TOK': sum(ctx_toks)/len(ctx_toks),
    }
    
    return quality_score(scores), scores


In [ ]:
qs, scores = benchmark_v0(k=5)
print(scores, "\nquality:", round(qs, 3))

{'COP': 0.18461538461538465, 'REC': 0.9710144927536232, 'CIP': 0.27318840579710146, 'RR': 0.9615384615384616, 'HR': 0.0, 'RT': 0.09394158308322613} 
quality: 0.605


In [14]:
import csv

OUT = r"/Users/alejandrogomez-paz/Desktop/RAG/baseline_old_rag_k5.csv"
row = {"run": "old_rag", "k": 5, "quality": round(qs, 4),
       **{m: round(v, 4) for m, v in scores.items()}}

with open(OUT, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    w.writeheader()
    w.writerow(row)

print("saved:", OUT)

saved: /Users/alejandrogomez-paz/Desktop/RAG/baseline_old_rag_k5.csv


In [ ]:
import json
import sys
import math
import re
import statistics
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/local_LLM")
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1")

# NOTE: `retrieve` and `rag_answer` were USED but never imported in the original.
# Importing them explicitly (adjust names if they live elsewhere in rad_ai).
from rad_ai import query, retrieve, rag_answer
from benchmark_golden_pairs_1 import golden_pairs, should_refuse_map, answer_keys
import time

'''
Quality:
  CoP  - Context Precision: fraction of retrieved chunks that are golden
  CoPg - Context Precision @ golden-count: correct / min(k, #golden)  (fixes the
         structural cap where 1 golden + k=5 makes max CoP = 0.2)
  nDCG - normalized DCG@k over binary golden relevance (rank-aware retrieval quality)
  REC  - Context Recall (Hit@k / recall over the golden set)
  CiP  - Citation Precision: fraction of cited sources that are correct
  CiR  - Citation Recall: fraction of golden sources that got cited
  CiF1 - Citation F1 (harmonic mean of CiP, CiR)
  RR   - Refusal Rate: fraction of correct refusal decisions (maximize)
  AC   - Answer Correctness vs answer_keys on answerable queries (maximize)
  HR   - Hallucination Rate: fraction of answers with unsupported claims (minimize)
Latency:
  RT   - mean retrieval time (s)
  GT   - mean generation time (s)          [NEW: was previously unmeasured]
  ET   - mean end-to-end time (s)           [NEW]
Cost:
  CTX_TOK - mean context tokens sent to the LLM

All aggregate scores are reported as mean +/- 95% CI across N_RUNS repeats
(LLM generation + judge are nondeterministic; a single run is an anecdote).
'''

CORPUS_PATH = r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1/corpus_1.jsonl"
REFUSAL = "don't have enough information"

QUALITY_WEIGHTS = {
    'AC':   0.30,   # did it actually answer correctly
    'CIF1': 0.20,   # cited the right sources (precision AND recall)
    'REC':  0.15,   # did retrieval surface the golden context at all
    'COPG': 0.10,   # golden-count-normalized precision
    'RR':   0.15,   # correct refusal decisions
    'HR':   0.10,   # 1 - hallucination rate
}

# Harder-benchmark knobs. Lower k / add adversarial sets to reveal real headroom.
N_RUNS = 3          # repeats for variance / confidence intervals
DEFAULT_K = 5
K_SWEEP = (3, 5, 8) # for benchmark_k_sweep()


def context_precision(query, retrieved_hits, gsp):
    if not retrieved_hits:
        return 0.0
    g = gsp.get(query, set())
    return sum(1 for h in retrieved_hits if get_citation(h) in g) / len(retrieved_hits)

def context_precision_at_g(query, retrieved_hits, gsp):
    """Correct / min(k, #golden). Full credit when all golden are retrieved,
    removing the structural cap that punishes single-golden queries."""
    g = gsp.get(query, set())
    if not g or not retrieved_hits:
        return None if not g else 0.0
    got = {get_citation(h) for h in retrieved_hits}
    correct = len(got & g)
    denom = min(len(retrieved_hits), len(g))
    return correct / denom if denom else 0.0

def context_recall(query, retrieved_hits, gsp):     # Hit@k / recall over golden set
    g = gsp.get(query, set())
    if not g:
        return None
    got = {get_citation(h) for h in retrieved_hits}
    return len(got & g) / len(g)

def ndcg_at_k(query, retrieved_hits, gsp):
    """Rank-aware retrieval quality, binary relevance from the golden set."""
    g = gsp.get(query, set())
    if not g:
        return None
    dcg = 0.0
    seen = set()  # credit each golden citation at most once (distinct chunks can share a source|page)
    for i, h in enumerate(retrieved_hits):
        c = get_citation(h)
        if c in g and c not in seen:
            seen.add(c)
            dcg += 1.0 / math.log2(i + 2)
    ideal_hits = min(len(g), len(retrieved_hits))
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg else 0.0

def citation_precision(cited_ids, golden_ids):
    if not cited_ids:
        return 0.0
    return sum(1 for c in cited_ids if c in golden_ids) / len(cited_ids)

def citation_recall(cited_ids, golden_ids):
    if not golden_ids:
        return None
    cited = set(cited_ids)
    return sum(1 for gc in golden_ids if gc in cited) / len(golden_ids)

def citation_f1(cited_ids, golden_ids):
    p = citation_precision(cited_ids, golden_ids)
    r = citation_recall(cited_ids, golden_ids)
    if r is None:
        return None
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

def detect_refusal(answer) -> bool:
    """Prefer a structured refusal flag if the pipeline returns a dict.
    Fall back to phrase matching for plain-string answers."""
    if isinstance(answer, dict):
        if 'refused' in answer:
            return bool(answer['refused'])
        text = answer.get('answer') or answer.get('text') or answer.get('response') or ''
    else:
        text = str(answer)
    t = text.lower()
    refusal_phrases = (
        REFUSAL,
        "not enough information",
        "cannot answer",
        "can't answer",
        "no relevant",
        "unable to find",
    )
    return any(p in t for p in refusal_phrases)

def refusal_score(did_refuse, should_refuse):
    correct = sum(1 for d, s in zip(did_refuse, should_refuse) if d == s)
    return correct / len(should_refuse)


def _answer_text(answer) -> str:
    if isinstance(answer, dict):
        return str(answer.get('answer') or answer.get('text')
                   or answer.get('response') or '')
    return str(answer)

def _normalize(s: str):
    return re.findall(r"[a-z0-9]+", s.lower())

def answer_correctness(answer, key) -> float:
    """Score an answer against its gold key in [0,1]. Robust to key shape:
      - str                       -> token-F1 overlap with the gold answer
      - list[str]                 -> fraction of required phrases present
      - dict{'must_include':[..]} -> fraction of required phrases present
      - dict{'answer': str}       -> token-F1 vs that gold answer
    A refusal on an answerable query scores 0."""
    if key is None:
        return None
    if detect_refusal(answer):
        return 0.0
    ans = _answer_text(answer)
    if not ans.strip():
        return 0.0

    # required-phrase style
    required = None
    gold_text = None
    if isinstance(key, dict):
        if key.get('must_include'):
            required = list(key['must_include'])
        elif key.get('keywords'):
            required = list(key['keywords'])
        elif key.get('answer'):
            gold_text = str(key['answer'])
    elif isinstance(key, (list, tuple, set)):
        required = list(key)
    elif isinstance(key, str):
        gold_text = key

    if required is not None:
        if not required:
            return None
        low = ans.lower()
        return sum(1 for p in required if str(p).lower() in low) / len(required)

    if gold_text is not None:
        gt = _normalize(gold_text)
        at = _normalize(ans)
        if not gt or not at:
            return 0.0
        gset, aset = set(gt), set(at)
        overlap = len(gset & aset)
        if overlap == 0:
            return 0.0
        p = overlap / len(aset)
        r = overlap / len(gset)
        return 2 * p * r / (p + r)
    return None


def context_tokens(hits) -> int:
    blocks = [
        f"[SOURCE: {h['source']} | page {h['page']} | score 0.000]\n{h['text']}"
        for h in hits
    ]
    context = "\n\n---\n\n".join(blocks)
    return len(context) // 4


def hallucination_rate(judgments):
    if not judgments:
        return 0.0
    return sum(judgments) / len(judgments)

def judge_hallucination(q, answer, chunks):
    """LLM-as-judge. Returns True if the answer makes claims not supported by the
    retrieved chunks. (Left as-is; scheduled to be fixed last.)"""
    import json, re
    from rad_ai import query

    if isinstance(answer, dict) and answer.get('refused'):
        return False

    if isinstance(answer, dict):
        ans = answer.get('answer') or answer.get('text') or answer.get('response') or json.dumps(answer)
    else:
        ans = str(answer)
    if not str(ans).strip():
        return False

    def chunk_text(h):
        for k in ('text', 'content', 'chunk', 'body', 'passage'):
            if isinstance(h, dict) and h.get(k):
                return str(h[k])
        return str(h)
    context = "\n\n".join(f"[{i+1}] {chunk_text(h)}" for i, h in enumerate(chunks)) or "(no context)"

    prompt = (
        "You are a strict factuality judge for a retrieval-augmented QA system. "
        "Given a QUESTION, the retrieved CONTEXT, and an ANSWER, decide whether the ANSWER "
        "contains a hallucination: any factual claim NOT supported by the CONTEXT. "
        "Judge support only against the CONTEXT, not outside knowledge. A refusal is NOT a hallucination.\n"
        'Respond with ONLY JSON: {"hallucinated": true|false}\n\n'
        f"QUESTION:\n{q}\n\nCONTEXT:\n{context}\n\nANSWER:\n{ans}\n"
    )

    try:
        raw = query(prompt)
        raw = raw if isinstance(raw, str) else str(raw)
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            return bool(json.loads(m.group(0)).get('hallucinated', False))
        return bool(re.search(r'\btrue\b|\byes\b', raw, re.I))
    except Exception:
        return False  # judge failed -> don't penalize


def quality_score(s):
    """Weighted composite over named, normalized weights. HR is inverted
    (1 - HR). Missing metrics are dropped and weights renormalized."""
    parts = {
        'AC':   s.get('AC'),
        'CIF1': s.get('CIF1'),
        'REC':  s.get('REC'),
        'COPG': s.get('COPG'),
        'RR':   s.get('RR'),
        'HR':   (1 - s['HR']) if s.get('HR') is not None else None,
    }
    num = 0.0
    wsum = 0.0
    for k, v in parts.items():
        if v is None:
            continue
        w = QUALITY_WEIGHTS.get(k, 0.0)
        num += w * v
        wsum += w
    return num / wsum if wsum else 0.0


def _mean(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else 0.0


# Multi-run benchmark with 95% confidence intervals

def _ci95(xs):
    """Mean and half-width of a 95% CI (normal approx). 0 width for n<2."""
    if not xs:
        return 0.0, 0.0
    m = statistics.mean(xs)
    if len(xs) < 2:
        return m, 0.0
    sd = statistics.stdev(xs)
    return m, 1.96 * sd / math.sqrt(len(xs))

def benchmark_v0(k=DEFAULT_K, n_runs=N_RUNS, hard=False):
    """Run the eval n_runs times; report per-metric mean +/- 95% CI.
    Returns (quality_mean, quality_ci, agg) where agg[metric] = (mean, ci)."""
    run_quals, run_scores = [], []
    for _ in range(n_runs):
        qs, sc = run_once(k=k, hard=hard)
        run_quals.append(qs)
        run_scores.append(sc)

    keys = run_scores[0].keys()
    agg = {}
    for key in keys:
        vals = [s[key] for s in run_scores]
        agg[key] = _ci95(vals)
    q_mean, q_ci = _ci95(run_quals)
    return q_mean, q_ci, agg

def benchmark_k_sweep(ks=K_SWEEP, n_runs=N_RUNS, hard=False):
    """Lower k = harder / less retrieval slack. Sweep to expose the tradeoff."""
    out = {}
    for k in ks:
        out[k] = benchmark_v0(k=k, n_runs=n_runs, hard=hard)
    return out

def _fmt(mean, ci):
    return f"{mean:.3f} +/- {ci:.3f}"

if __name__ == "__main__":
    q_mean, q_ci, agg = benchmark_v0(k=DEFAULT_K, n_runs=N_RUNS, hard=False)
    print(f"n_runs={N_RUNS}  k={DEFAULT_K}")
    for key, (m, ci) in agg.items():
        print(f"  {key:8s} {_fmt(m, ci)}")
    print("quality:", _fmt(q_mean, q_ci))

In [17]:
toks = [context_tokens([{"source": m["source"], "page": m["page"],
                         "chunk_id": m["chunk_id"], "text": txt}
                        for _, txt, m in retrieve(q, k=5)])
        for q in should_refuse_map]
print("CTX_TOK:", sum(toks) / len(toks))

CTX_TOK: 1342.923076923077


In [21]:
def chunk_token_budget(num_ctx=2048, gen_reserve=300, k=5,
                       est=lambda s: len(s) // 4):
    """Worst-case token budget available for retrieved chunk TEXT,
    after every other prompt component takes its share."""
    SYSTEM = ("You are RAD-AI, a radiation safety assistant.\n"
              "Use ONLY the provided context to answer.\n"
              "If the answer is not in the context, say: "
              "'I don't have enough information in the provided files.'\n"
              "Cite sources as (source, page).")

    corpus = [r for r in (json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8"))
              if r.get("record_type") == "chunk"]

    worst_hdr   = max(est(f"[SOURCE: {r['source']} | page {r['section']} | score 0.000]\n")
                      for r in corpus)
    worst_query = max(est(q) for q in should_refuse_map)

    overhead = (est(SYSTEM)
                + est("\n\nCONTEXT:\n") + est("\n\nQUESTION:\n")
                + k * worst_hdr                    # one header per hit
                + (k - 1) * est("\n\n---\n\n")     # separators
                + worst_query
                + 50                               # chat-template wrapper
                + gen_reserve)                     # room for the answer

    budget = num_ctx - overhead
    print(f"num_ctx={num_ctx} | overhead={overhead} | "
          f"chunk budget={budget} total, {budget // k}/chunk at k={k}")
    return budget

print(chunk_token_budget())

num_ctx=2048 | overhead=664 | chunk budget=1384 total, 276/chunk at k=5
1384
